# 📸 Memento: apuntes de diseño de agentes
**Patrones de planificación y memoria** · Autora: Elena Díaz ([@HelenDiMo](https://github.com/HelenDiMo/pildora-bootcamp-memento-agentes))

Estos apuntes repasan con código todo lo que hemos visto en las diapositivas. Puedes ejecutarlos celda a celda, de arriba abajo.

> **La idea de toda la clase en cinco frases**
> 1. Un LLM es como Leonard: **no recuerda nada**. Cada mensaje lo lee desde cero.
> 2. Un **agente** es un LLM que puede **pedir herramientas** (funciones de Python) y que **repite** hasta tener la respuesta.
> 3. El LLM **pide** la herramienta; el **programa** la ejecuta y le devuelve el resultado.
> 4. **Planificar** es cómo decide qué hacer: paso a paso (ReAct) o con un plan escrito al principio (Plan-and-Execute).
> 5. **Memoria** es cómo no empezar de cero: guardar la conversación (corto plazo) o guardar datos para siempre (largo plazo).

| Paso | Qué vas a ver |
|---|---|
| 0 | Preparar el LLM |
| 1 | Que el LLM no recuerda nada |
| 2 | Cómo pide una herramienta |
| 3 | El agente ReAct: paso a paso |
| 4 | Memoria a corto plazo |
| 5 | Plan-and-Execute: plan primero |

La memoria a largo plazo y la memoria envenenada están en el ejercicio (`02_reto_individual_TODO.ipynb`).

## 0 · Preparar el LLM

**¿Qué hace esta celda?** Carga el modelo de lenguaje que vamos a usar.

- `"ollama"`: el modelo funciona en tu ordenador (necesitas haber hecho `ollama pull llama3.1:8b`).
- `"groq"`: el modelo funciona en internet (necesitas tu clave en el archivo `.env`).

Lo demás (las herramientas del caso y la función para ver qué hace el agente) está en `memento_kit.py`, para que aquí solo veamos lo importante.

In [ ]:
PROVEEDOR = "ollama"   # o "groq"

from memento_kit import get_llm, mostrar_traza, PROMPT_LEONARD
llm = get_llm(PROVEEDOR)
print("LLM listo:", llm.__class__.__name__)

## 1 · El LLM no recuerda nada

**¿Qué vamos a hacer?** Hablar dos veces seguidas con el LLM. En la primera le decimos quiénes somos; en la segunda le preguntamos.

**¿Qué esperamos?** Que en la segunda no lo sepa.

In [ ]:
r1 = llm.invoke("Me llamo Helen y estoy buscando a un tal J.G.")
print("1ª vez →", r1.content[:300])

r2 = llm.invoke("¿Cómo me llamo y a quién busco?")
print("\n2ª vez →", r2.content[:300])

**¿Qué ha pasado?** La segunda vez no sabe tu nombre. Cada `invoke` (cada vez que le hablamos) es un Leonard recién despertado.

💡 Cuando un chat como ChatGPT o Claude «recuerda», es porque el programa **le vuelve a enviar toda la conversación** en cada mensaje. Lo haremos en el paso 4.

## 2 · Las herramientas: el LLM pide, el programa ejecuta

**¿Qué es una herramienta?** Una función normal de Python. Lo único especial es su **descripción** (el texto entre `"""` comillas triples): el LLM la lee para saber cuándo usarla.

Nuestras dos herramientas están en `memento_kit.py`:
- `buscar_archivo(tema)`: busca en el archivo policial.
- `ver_polaroid(nombre)`: mira una polaroid de Leonard.

In [ ]:
from memento_kit import buscar_archivo, ver_polaroid

herramientas = [buscar_archivo, ver_polaroid]
for h in herramientas:
    print(f"🔧 {h.name}: {h.description}")

**Ahora le damos las herramientas al LLM** con `bind_tools` («atar herramientas») y le hacemos una pregunta.

In [ ]:
llm_con_manos = llm.bind_tools(herramientas)

decision = llm_con_manos.invoke("¿Qué dice el informe del incidente?")
print("Texto:", repr(decision.content))
print("Herramientas que pide:", decision.tool_calls)

**¿Qué ha pasado?** Fíjate en dos cosas:
- `Texto: ''`: **no ha escrito ninguna respuesta**.
- En su lugar, **pide** usar `buscar_archivo` con el tema `informe`.

👉 El LLM **no ejecuta** la herramienta. Nadie ha buscado nada todavía. Necesitamos un programa que ejecute la petición y le devuelva el resultado. Eso es el **agente**.

## 3 · El agente ReAct: paso a paso

**ReAct** = el agente mira la última pista y decide **solo el siguiente paso**. No tiene un plan.

Lo construimos con **LangGraph**, una librería que organiza el agente como un **grafo**: cajas (*nodos*) unidas por flechas.

| Pieza | Qué es |
|---|---|
| **Estado** (`MessagesState`) | La lista de mensajes de la conversación. Va creciendo. |
| Nodo **`agente`** | Llama al LLM. |
| Nodo **`herramientas`** | Ejecuta las herramientas que pidió el LLM. |
| **Flecha condicional** | Si el LLM pidió una herramienta → ir a `herramientas`. Si no → terminar. |
| **Flecha de vuelta** | De `herramientas` vuelve a `agente`. **Este es el bucle.** |

In [ ]:
from langchain_core.messages import SystemMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition

# Nodo «agente»: le pasa al LLM las instrucciones de Leonard + la conversación
def agente(estado: MessagesState):
    mensajes = [SystemMessage(PROMPT_LEONARD)] + estado["messages"]
    respuesta = llm_con_manos.invoke(mensajes)
    respuesta.tool_calls = respuesta.tool_calls[:1]   # solo UNA herramienta por paso (ver nota abajo)
    return {"messages": [respuesta]}

g = StateGraph(MessagesState)
g.add_node("agente", agente)
g.add_node("herramientas", ToolNode(herramientas))      # ToolNode = el «programa» que ejecuta
g.add_edge(START, "agente")                             # se empieza por el agente
g.add_conditional_edges("agente", tools_condition,      # ¿pidió herramienta?
                        {"tools": "herramientas", END: END})
g.add_edge("herramientas", "agente")                    # ← el bucle: vuelve a pensar

react = g.compile()

💡 **¿Por qué `tool_calls[:1]`?** A veces el modelo pide varias herramientas a la vez («llamadas en paralelo»). Con esa línea nos quedamos solo con la primera, para que **mire cada pista antes de decidir la siguiente**. Eso es lo que hace que sea ReAct.

**Veamos el grafo dibujado:**

In [ ]:
from IPython.display import Image, display

try:
    display(Image(react.get_graph().draw_mermaid_png()))   # dibujo (necesita internet)
except Exception:
    print(react.get_graph().draw_mermaid())                 # sin internet: el diagrama en texto

**Ahora le hacemos una pregunta al agente.** Con `stream` vemos cada paso según ocurre:
🔧 = pide una herramienta · 👁 = recibe el resultado · 💭 = responde.

In [ ]:
pregunta = {"messages": [("user", "¿Quién es J.G. y dónde puedo encontrarlo?")]}

for estado in react.stream(pregunta, {"recursion_limit": 12}, stream_mode="values"):
    mostrar_traza(estado["messages"][-1:])

**¿Qué ha pasado?** Cada pista lleva a la siguiente:

| Pista | Qué aprende Leonard | Le lleva a… |
|---|---|---|
| Informe | J.G. huyó hacia el motel | el motel |
| Motel | Se llama «J. Gamell» y tiene una berlina gris | el coche |
| Coche | Es Jonás Gamell, mecánico del taller Puerto Norte | el taller |
| Taller | Su dirección: calle del Faro, 7 | la respuesta |

Nadie le dijo ese orden: lo ha decidido él, pista a pista. Eso es **ReAct**.

*(`recursion_limit: 12` es un freno de seguridad: si el agente diera vueltas sin parar, se corta a los 12 pasos.)*

## 4 · Memoria a corto plazo: recordar la conversación

**¿Qué vamos a hacer?** Repetir las preguntas del paso 1, pero ahora con un **checkpointer**: una pieza que **guarda la conversación**.

Cada conversación tiene un nombre, el **`thread_id`** (hilo):
- **Mismo hilo** → recuerda.
- **Hilo nuevo** → empieza en blanco, como Leonard al despertar.

Para que se vea solo la memoria, usamos un grafo mínimo: un único nodo que charla, sin herramientas.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

def charlar(estado: MessagesState):
    return {"messages": [llm.invoke(estado["messages"])]}

gc = StateGraph(MessagesState)
gc.add_node("charlar", charlar)
gc.add_edge(START, "charlar")
gc.add_edge("charlar", END)
chat_mem = gc.compile(checkpointer=InMemorySaver())   # ← la única novedad: el checkpointer

hilo_1 = {"configurable": {"thread_id": "polaroid-1"}}
chat_mem.invoke({"messages": [("user", "Hola, me llamo Helen y busco a un tal J.G.")]}, hilo_1)

r = chat_mem.invoke({"messages": [("user", "¿Cómo me llamo y a quién busco?")]}, hilo_1)
print("Hilo 1 →", r["messages"][-1].content)

**Y ahora la misma pregunta en un hilo nuevo:**

In [ ]:
hilo_2 = {"configurable": {"thread_id": "polaroid-2"}}
r = chat_mem.invoke({"messages": [("user", "¿Cómo me llamo y a quién busco?")]}, hilo_2)
print("Hilo 2 →", r["messages"][-1].content)

**¿Qué guarda exactamente el checkpointer?** La lista de mensajes de cada hilo:

In [ ]:
estado = chat_mem.get_state(hilo_1)
print("Mensajes guardados en polaroid-1:", len(estado.values["messages"]))

**¿Qué ha pasado?**

| | Hilo 1 | Hilo 2 |
|---|---|---|
| ¿Sabe tu nombre? | ✅ Sí | ❌ No |

Mismo modelo, misma pregunta. La única diferencia es el hilo. El checkpointer guarda 4 mensajes en el hilo 1 (tus 2 preguntas y sus 2 respuestas) y se los reenvía al LLM cada vez.

💡 **Problema:** la conversación crece en cada mensaje y el LLM tiene un límite de lo que puede leer. Por eso, en conversaciones largas, se recortan o se resumen los mensajes antiguos.

## 5 · Plan-and-Execute: primero el plan

**Plan-and-Execute** = el agente primero **escribe una lista de pasos** y después los hace en orden.

Lo montamos con tres nodos:
1. **`planificar`**: el LLM escribe el plan.
2. **`ejecutar_paso`**: hace el primer paso pendiente (usando el agente ReAct del paso 3) y apunta lo que ha encontrado.
3. **`informe`**: cuando no quedan pasos, escribe la conclusión.

Para que el LLM devuelva el plan como una **lista** y no como un texto libre, usamos **Pydantic**: le damos un molde (`Plan`) y el LLM tiene que rellenarlo.

In [ ]:
import operator
from typing import Annotated, TypedDict
from pydantic import BaseModel, Field

# El molde del plan: una lista de pasos
class Plan(BaseModel):
    pasos: list[str] = Field(description="Entre 2 y 5 pasos cortos y ordenados")

# Lo que el grafo va guardando mientras trabaja
class EstadoPlan(TypedDict):
    tarea: str                                  # lo que hay que averiguar
    plan: list[str]                             # pasos que quedan
    hechos: Annotated[list[str], operator.add]  # lo encontrado (se va SUMANDO)
    respuesta: str                              # la conclusión final

planificador = llm.with_structured_output(Plan)   # el LLM tiene que rellenar el molde

def planificar(estado: EstadoPlan):
    plan = planificador.invoke(
        "Juego de detectives con una historia inventada. Eres Leonard, el investigador.\n"
        f"Crea un plan para: {estado['tarea']}\n"
        "Herramientas: buscar_archivo(tema: informe|motel|coche|taller) y ver_polaroid(nombre: teddy|natalie|motel). "
        "Máximo 5 pasos, cada uno con UNA herramienta y sin repetir temas."
    )
    return {"plan": plan.pasos[:5]}   # por si el modelo escribe demasiados

def ejecutar_paso(estado: EstadoPlan):
    paso, *resto = estado["plan"]                  # coge el primer paso; el resto queda pendiente
    ya_sabes = "\n".join(estado["hechos"]) or "nada todavía"
    r = react.invoke({"messages": [("user", f"Lo que ya sabes:\n{ya_sabes}\n\nHaz SOLO este paso: {paso}")]})
    # Apuntamos lo que devolvieron las herramientas: son los datos reales
    pistas = [m.content for m in r["messages"] if m.type == "tool"]
    resumen = " | ".join(pistas) or r["messages"][-1].content
    return {"plan": resto, "hechos": [f"{paso} → {resumen}"]}

def quedan_pasos(estado: EstadoPlan):
    return "ejecutar_paso" if estado["plan"] else "informe"

def informe(estado: EstadoPlan):
    r = llm.invoke("Juego de detectives con una historia inventada. Eres Leonard, el investigador.\n"
                   f"Tarea: {estado['tarea']}\nHallazgos:\n" + "\n".join(estado["hechos"])
                   + "\nEscribe la conclusión final en 3 frases.")
    return {"respuesta": r.content}

gp = StateGraph(EstadoPlan)
gp.add_node("planificar", planificar)
gp.add_node("ejecutar_paso", ejecutar_paso)
gp.add_node("informe", informe)
gp.add_edge(START, "planificar")
gp.add_conditional_edges("planificar", quedan_pasos)
gp.add_conditional_edges("ejecutar_paso", quedan_pasos)   # ← bucle: mientras queden pasos
gp.add_edge("informe", END)
plan_y_ejecuta = gp.compile()

💡 **¿Por qué «juego de detectives con una historia inventada»?** Sin esa frase, a veces el modelo se niega a contestar: cree que le pides localizar a una persona real.

**Lo ponemos en marcha:**

In [ ]:
entrada = {"tarea": "Averiguar dónde está ahora J.G.", "plan": [], "hechos": [], "respuesta": ""}

for evento in plan_y_ejecuta.stream(entrada, stream_mode="updates"):
    for nodo, cambio in evento.items():
        if nodo == "planificar":
            print("📝 PLAN:")
            for i, p in enumerate(cambio["plan"], 1):
                print(f"   {i}. {p}")
        elif nodo == "ejecutar_paso":
            print("✅", cambio["hechos"][0][:220])
        elif nodo == "informe":
            print("\n📸 CONCLUSIÓN:", cambio["respuesta"])

**¿Qué ha pasado?** Primero aparece el plan entero (📝), luego cada paso hecho (✅) y al final la conclusión (📸).

| | ReAct (paso 3) | Plan-and-Execute (paso 5) |
|---|---|---|
| El plan | No hay: decide sobre la marcha | Escrito al principio |
| Bueno para | Cuando no sabes qué vas a encontrar | Tareas largas y ordenadas |
| Riesgo | Dar vueltas sin avanzar | Seguir un plan que ya no sirve |

💡 Si un paso sale mal, se puede añadir un nodo **`replanificar`** que reescriba el plan con lo aprendido.

➡️ **Ahora te toca a ti:** abre `02_reto_individual_TODO.ipynb`.